# Clustering: K-Means y DBSCAN

El **clustering** es una potente técnica de aprendizaje no supervisado que tiene como objetivo agrupar puntos de datos similares en función de sus características o atributos.

Podemos utilizarla para obtener información sobre la **estructura subyacente** de conjuntos de datos grandes y complejos, así como para identificar patrones y relaciones que pueden no ser evidentes a primera vista.

En este notebook exploraremos dos algoritmos de clustering diferentes: KMeans y DBSCAN

Estos algoritmos se utilizan ampliamente en diversos campos, incluida la química, y pueden ayudar a los investigadores a comprender mejor los datos y tomar decisiones fundamentadas.

En este notebook haremos lo siguiente:

- Generar un conjunto de datos sintético (un ejemplo de juguete).

- Aplicar cada algoritmo a ese conjunto de datos sintético.

- Evaluar el rendimiento utilizando métricas de clustering.

- Aplicar todo esto a un conjunto de datos real de química.

En el último ejercicio encontrarás el número óptimo de clusters, utilizando métricas como silhouette e inertia.

In [ ]:
!pip install rdkit

In [ ]:
from sklearn.datasets import make_blobs
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd

from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from sklearn.metrics import pairwise_distances
from rdkit.Chem import AllChem, Descriptors, MolFromSmiles, rdMolDescriptors
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

### Generación de datos sintéticos

En este paso generamos un conjunto de datos sintético que simula la estructura de datos de huellas moleculares (*molecular fingerprints*).

Una huella molecular es un **vector binario de alta dimensionalidad** en el que cada posición indica la presencia o ausencia de determinados patrones estructurales dentro de la molécula. Estos patrones pueden corresponder, por ejemplo, a subestructuras químicas, tipos de enlaces o configuraciones locales de átomos.

De forma simplificada, cada bit del vector toma el valor:

- **1** si la molécula contiene ese patrón estructural.
- **0** si no está presente.

Por ejemplo, el vector [1,0,1,...] puede representar que la molécula SÍ tiene anillo aromático (1), NO tiene grupo OH (0), SÍ tiene una cadena larga (1), etc.

Este tipo de representación permite convertir estructuras químicas complejas en vectores que pueden ser comparados y analizados mediante técnicas de aprendizaje automático, como búsqueda de similitud, clasificación o clustering.

En muchos casos, estas huellas tienen **centenares o miles de dimensiones** (por ejemplo, 512, 1024 o 2048 bits), lo que da lugar a datos binarios de alta dimensionalidad. En este notebook se utiliza una representación binaria sintética para emular este tipo de datos y poder ilustrar el funcionamiento de distintos algoritmos de clustering.

Para ello, creamos varios clusters con distintas cantidades de muestras y diferentes niveles de dispersión. En nuestro caso, cada punto tendrá 512 dimensiones, lo que representa vectores de características de alta dimensionalidad similares a los utilizados para describir moléculas.

In [ ]:
# Definimos características de los clusters sintéticos
n_clusters = 4
n_features = 512

# Generaremos 4 clusters con diferentes desviaciones estándar y número de puntos
cluster_stds = [0.5, 1.5, 1, 2.0]
n_samples = [200, 300, 100, 150]
cluster_centers = np.random.randint(-5, 5, size=(n_clusters, n_features))

X, y = make_blobs(n_samples=n_samples, centers=None, cluster_std=cluster_stds, n_features=n_features, center_box=(-1, 1))

# Convertimos a binario (simulando fingerprints)
X_binary = np.where(X > 0, 1, 0)

### Reducción de dimensionalidad mediante PCA

Dado que los datos tienen 512 dimensiones, resulta difícil visualizarlos directamente. Para facilitar su interpretación, aplicamos **Principal Component Analysis (PCA)** para proyectar los datos a un espacio de menor dimensionalidad.

En este caso se reducen a tres componentes principales, que capturan la mayor parte de la variabilidad del conjunto de datos. Estas coordenadas se utilizarán posteriormente para representar gráficamente los clusters.

In [ ]:
pca = PCA(n_components=3)
coords = pca.fit_transform(X_binary)

### Clustering con K-Means

A continuación aplicamos el algoritmo **K-Means** para agrupar las observaciones.

K-Means es un método de clustering basado en centroides que divide los datos en un número predefinido de clusters. El algoritmo busca minimizar la distancia entre cada punto y el centroide del cluster al que pertenece.

Se especifica el número de clusters esperado y se ejecuta el algoritmo sobre los datos binarios generados previamente.

In [ ]:
# Defining KMeans method with n_clusters
kmeans = KMeans(n_clusters=n_clusters, n_init=10, init='k-means++')
kmeans.fit(X_binary)

Ahora, se comparan los clusters reales del conjunto de datos sintético con los clusters predichos por el algoritmo K-Means.

Utilizando las coordenadas obtenidas mediante PCA, se representan ambos resultados en dos gráficos. Esto permite evaluar visualmente hasta qué punto el algoritmo ha sido capaz de recuperar la estructura de clusters original del conjunto de datos.

In [ ]:
def plot_2d(X, y, title, ax):
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', s=50)
    ax.set_title(title)
    legend = ax.legend(*scatter.legend_elements(), loc="best", title="Clusters")
    ax.add_artist(legend)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

plot_2d(coords, y, title="Clusters reales", ax=ax1)
plot_2d(coords, kmeans.labels_, title="Clusters predichos por KMeans", ax=ax2)

plt.show()

### Clustering con DBSCAN

En esta celda se aplica el algoritmo **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**.

A diferencia de K-Means, DBSCAN identifica clusters en función de la densidad de los datos. El algoritmo agrupa puntos que se encuentran dentro de una distancia máxima (`eps`) y que tienen al menos un número mínimo de vecinos (`min_samples`).

Este enfoque permite detectar clusters de forma arbitraria y también identificar puntos que no pertenecen a ningún cluster, tratándolos como ruido.

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5, metric='euclidean') # 'euclidean' para datos continuos, 'jaccard' para datos binarios
dbscan.fit(X_binary)

Representamos nuevamente los clusters reales y los clusters obtenidos por el algoritmo, esta vez utilizando DBSCAN.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

plot_2d(coords, y, title="Clusters reales", ax=ax1)
plot_2d(coords, dbscan.labels_, title="Clusters predichos por DBSCAN", ax=ax2)

plt.show()

### Evaluación del número de clusters en K-Means

En este paso se analiza cómo cambia el rendimiento de K-Means al variar el número de clusters.

Para distintos valores de `k`, se calculan dos métricas:

- **Inercia**: mide la suma de las distancias cuadradas entre cada punto y el centroide de su cluster. Valores más bajos indican clusters más compactos.
- **Silhouette Score**: evalúa la separación entre clusters teniendo en cuenta la cohesión interna y la distancia a otros clusters.

Estas métricas permiten analizar la estructura de los datos y ayudar a seleccionar un número adecuado de clusters.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('K-Means')
inertia = []
silhouette_scores = []

min_clusters = 2
max_clusters = 10

for n_clusters in range(min_clusters, max_clusters + 1):
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, init='k-means++', random_state=42)
    labels = kmeans.fit_predict(X_binary.astype(bool))
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_binary.astype(bool), labels))

ax1.plot(range(min_clusters, max_clusters + 1), inertia, label='k-Means')
ax2.plot(range(min_clusters, max_clusters + 1), silhouette_scores, label='k-Means')
ax1.set_xlabel("Número de clusters")
ax1.set_ylabel("Inercia")
ax1.legend()

ax2.set_xlabel("Número de clusters")
ax2.set_ylabel("Silhouette Score")
ax2.legend()

plt.show()

## Dataset químico

Ahora vamos a explorar cómo aplicar técnicas de clustering para agrupar moléculas según su similitud estructural. Para ello partimos de una colección de moléculas representadas mediante SMILES, una notación textual que describe la estructura química.

Nuestro objetivo será transformar estas moléculas en representaciones numéricas que capturen información relevante sobre su estructura, las llamadas *Morgan Fingerprints*.

A partir de estas representaciones aplicaremos algoritmos de clustering no supervisado, principalmente K-Means, con el objetivo de descubrir grupos de moléculas con características estructurales similares. Finalmente, analizaremos los clusters obtenidos y visualizaremos moléculas representativas de cada grupo para interpretar los resultados desde una perspectiva química.


### Carga y exploración inicial del dataset

En este primer paso cargamos el dataset que contiene las moléculas representadas mediante **SMILES (Simplified Molecular Input Line Entry System)**, una notación textual muy utilizada para describir estructuras químicas.

Una vez cargados los datos, hacemos una exploración rápida para entender qué tenemos entre manos:

* Observamos la **dimensión del dataset**.
* Inspeccionamos las **primeras filas** para ver cómo están representadas las moléculas.
* Revisamos las **columnas disponibles**.
* Contamos cuántos **SMILES válidos** hay y cuántos son **únicos**.

In [ ]:
data_ex = pd.read_csv("https://raw.githubusercontent.com/Jorgeprevi/ML_bootcamp_2026/refs/heads/main/clustering/moleculas.csv")

print("Shape del dataset:", data_ex.shape)
display(data_ex.head(10))

print("\nColumnas:")
print(data_ex.columns.tolist())

if "smiles" in data_ex.columns:
    print("\nNúmero de SMILES no nulos:", data_ex["smiles"].notna().sum())
    print("Número de SMILES únicos:", data_ex["smiles"].nunique())

### Visualización de algunas moléculas

Antes de empezar a modelar, es útil **mirar las moléculas** con las que estamos trabajando.

Seleccionamos un pequeño conjunto de SMILES del dataset y los convertimos en objetos moleculares utilizando **RDKit**. A partir de ahí generamos una cuadrícula de imágenes que nos permite visualizar varias moléculas a la vez.

Este paso nos ayuda a:

* Confirmar que los SMILES se están interpretando correctamente.
* Tener una intuición visual de las **estructuras químicas** presentes en el dataset.
* Observar si existen patrones estructurales comunes entre moléculas.

In [ ]:
sample = data_ex["smiles"].head(12).tolist()
sample_mols = [Chem.MolFromSmiles(smi) for smi in sample]

img = Draw.MolsToGridImage(
    sample_mols,
    molsPerRow=4,
    subImgSize=(220, 220),
    legends=[f"Mol {i+1}" for i in range(len(sample_mols))]
)

img

### Generación de fingerprints moleculares

Para aplicar algoritmos de clustering necesitamos convertir las moléculas en **vectores numéricos**. Para ello utilizamos **Morgan fingerprints**, una de las representaciones más utilizadas en química computacional.

Este tipo de fingerprint codifica la presencia de subestructuras químicas en un vector binario. En este caso generamos vectores de **1024 bits**, donde cada bit indica la presencia o ausencia de determinadas características estructurales.

Durante este proceso:

* Convertimos cada SMILES en una molécula RDKit.
* Generamos su fingerprint Morgan.
* Filtramos posibles moléculas inválidas.

Además mantenemos dos representaciones del fingerprint:

* una versión **numérica**, útil para algoritmos como K-Means
* una versión **booleana**, adecuada para métricas como **Jaccard**

Una vez obtenidas las representaciones, evaluamos distintos valores posibles del número de clusters (`k`). Para ello calculamos dos métricas:

* **Inercia**, que mide la compacidad de los clusters
* **Silhouette Score**, que evalúa la separación entre clusters

Esto nos ayuda a identificar valores razonables de `k` antes de realizar el clustering final.

In [ ]:
# Función para convertir SMILES a Morgan fingerprints
morgan_gen = GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_morgan(smiles):
    fps = []
    valid_idx = []

    for i, smi in enumerate(smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = morgan_gen.GetFingerprint(mol)
            fps.append(np.array(fp, dtype=np.int8))
            valid_idx.append(i)

    return np.array(fps), valid_idx

# Featurizar SMILES
X_ex, valid_idx = smiles_to_morgan(data_ex["smiles"])

# Por seguridad, nos quedamos solo con los válidos tras featurización
data_ex = data_ex.iloc[valid_idx].reset_index(drop=True)

# Versión booleana para Jaccard
X_ex_bool = X_ex.astype(bool)

min_clusters=2
max_clusters=10

kmeans = KMeans(n_init=10, random_state=42)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("KMeans")
inertia = []
silhouette_scores = []
for n_clusters in range(min_clusters, max_clusters + 1):
    kmeans.set_params(n_clusters=n_clusters)
    labels = kmeans.fit_predict(X_ex)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_ex, labels))
cluster_range = range(min_clusters, max_clusters + 1)
ax1.plot(cluster_range, inertia, marker="o")
ax2.plot(cluster_range, silhouette_scores, marker="o")
ax1.set_xlabel("Number of clusters")
ax1.set_ylabel("Inertia")
ax2.set_xlabel("Number of clusters")
ax2.set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

### Clustering de moléculas y selección de representantes

Una vez decidido el número de clusters, aplicamos **K-Means** sobre los fingerprints moleculares para agrupar las moléculas según su similitud estructural.

Después de entrenar el modelo, queremos entender **qué tipo de moléculas representa cada cluster**. Para ello identificamos, dentro de cada cluster, las moléculas más cercanas al **centroide** del cluster.

Estas moléculas pueden considerarse **representativas** del grupo, ya que son las que mejor capturan las características promedio del cluster.

Finalmente mostramos visualmente estas moléculas representativas utilizando RDKit. Esto nos permite interpretar los clusters desde una perspectiva química y observar si aparecen **patrones estructurales comunes** dentro de cada grupo.

In [ ]:
N_CLUSTERS = 3

kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42).fit(X_ex)

def plot_cluster_molecules(smiles_list, title, legends=None, mols_per_row=5):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(200, 200),
        # legends=legends
    )
    print(title)
    display(img)

# Representantes KMeans: seleccionamos las moléculas más cercanas a cada centroide
def get_kmeans_representatives(X, labels, centers, smiles, n_clusters, n_molecules=5):
    representatives = {}

    for i in range(n_clusters):
        cluster_idx = np.where(labels == i)[0]
        X_cluster = X[cluster_idx]

        dists = np.linalg.norm(X_cluster - centers[i], axis=1)
        nearest_idx_local = np.argsort(dists)[:n_molecules]
        nearest_idx_global = cluster_idx[nearest_idx_local]

        representatives[i] = nearest_idx_global

    return representatives

# Representantes KMeans: seleccionamos las moléculas más cercanas a cada centroide
kmeans_repr = get_kmeans_representatives(
    X_ex,
    kmeans.labels_,
    kmeans.cluster_centers_,
    data_ex["smiles"].values,
    N_CLUSTERS,
    n_molecules=5
)

print("KMeans clusters:")
for cluster_id, indices in kmeans_repr.items():
    smiles_sel = data_ex.iloc[indices]["smiles"].tolist()
    legends = [f"Cluster {cluster_id+1}"] * len(smiles_sel)
    plot_cluster_molecules(smiles_sel, f"Cluster {cluster_id+1}", legends=legends)